# Análise da população — Censos 2010 e 2022

Este notebook lê a tabela de municípios, agrega a população por estado, ordena pelo maior crescimento entre 2010 e 2022, faz a mesma análise por município e gera um gráfico dos 10 estados com maior crescimento absoluto.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
caminho = "CD2022_Populacao_2010_Compatibilizada_20231222 (1).xlsx"
df = pd.read_excel(caminho, sheet_name="Municípios", header=2)

df = df.rename(columns={
    "População 2010 (Alterações de Limites até 2022)1": "populacao_2010",
    "População Censo 2022": "populacao_2022",
    "NOME DO MUNICÍPIO": "municipio",
    "COD. UF": "cod_uf",
    "COD. MUNIC": "cod_municipio",
})

df["populacao_2010"] = pd.to_numeric(df["populacao_2010"], errors="coerce")
df["populacao_2022"] = pd.to_numeric(df["populacao_2022"], errors="coerce")
df = df.dropna(subset=["UF", "municipio", "populacao_2010", "populacao_2022"]).copy()

df.head()

In [ ]:

pop_estado = (
    df.groupby(["cod_uf", "UF"], as_index=False)
      .agg(
          populacao_2010=("populacao_2010", "sum"),
          populacao_2022=("populacao_2022", "sum"),
      )
)
pop_estado["crescimento_absoluto"] = pop_estado["populacao_2022"] - pop_estado["populacao_2010"]
pop_estado["crescimento_percentual"] = pop_estado["crescimento_absoluto"] / pop_estado["populacao_2010"] * 100
pop_estado = pop_estado.sort_values("crescimento_absoluto", ascending=False).reset_index(drop=True)
pop_estado

In [ ]:

pop_estado.to_csv("populacao_por_estado.csv", sep=";", index=False, encoding="utf-8-sig")

In [ ]:

pop_municipio = df[[
    "cod_uf", "UF", "cod_municipio", "municipio", "populacao_2010", "populacao_2022"
]].copy()
pop_municipio["crescimento_absoluto"] = pop_municipio["populacao_2022"] - pop_municipio["populacao_2010"]
pop_municipio["crescimento_percentual"] = pop_municipio["crescimento_absoluto"] / pop_municipio["populacao_2010"] * 100
pop_municipio = pop_municipio.sort_values("crescimento_absoluto", ascending=False).reset_index(drop=True)
pop_municipio.head(20)

In [ ]:

pop_municipio.to_csv("populacao_por_municipio.csv", sep=";", index=False, encoding="utf-8-sig")

In [ ]:

top = pop_estado.head(10).melt(
    id_vars="UF",
    value_vars=["populacao_2010", "populacao_2022"],
    var_name="ano",
    value_name="populacao",
)
top["ano"] = top["ano"].replace({"populacao_2010": "2010", "populacao_2022": "2022"})

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=top, x="UF", y="populacao", hue="ano", ax=ax)
ax.set_title("10 UFs com maior crescimento absoluto: 2010 vs 2022")
ax.set_xlabel("UF")
ax.set_ylabel("Habitantes")
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig("grafico_top_10_estados.png", dpi=150, bbox_inches="tight")
plt.show()